In [7]:
import os
import re
import requests
import pandas as pd
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from datetime import datetime, timezone

# 1. 환경변수 로드
load_dotenv(override=True)
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")

# 2. HTML 특수문자 및 태그 정제 함수
def clean_html(text):
    clean_text = re.sub(r'<.*?>', '', text)
    clean_text = BeautifulSoup(clean_text, "html.parser").text
    return clean_text.strip()

# 3. 특정 동의 상권/외식 뉴스 검색 함수
def fetch_dong_news(dong_name, keyword="외식 상권", display=50):
    url = "https://openapi.naver.com/v1/search/news.json"
    headers = {
        "X-Naver-Client-Id": CLIENT_ID,
        "X-Naver-Client-Secret": CLIENT_SECRET,
    }
    
    query = f"{dong_name} {keyword}"
    params = {
        "query": query,
        "display": display,
        "sort": "sim"  # 관련도순(sim) 또는 최신순(date)
    }
    
    response = requests.get(url, headers=headers, params=params)
    
    if not response.ok:
        print(f"[{dong_name}] 호출 실패: {response.status_code}")
        return []
    
    items = response.json().get("items", [])
    news_list = []
    
    # 4. 광고성/분양 노이즈 필터링
    noise_keywords = ['분양', '모델하우스', '체험단', '협찬', '입주자']
    
    for item in items:
        title = clean_html(item.get("title", ""))
        description = clean_html(item.get("description", ""))
        
        if any(noise in title or noise in description for noise in noise_keywords):
            continue
            
        news_list.append({
            "dong": dong_name,
            "title": title,
            "description": description,
            "originallink": item.get("originallink", ""),
            "link": item.get("link", ""),
            "pubDate": item.get("pubDate", "")
        })
        
    return news_list

# 5. 여러 행정동 일괄 수집
target_dongs = ["망원동", "성수동", "연남동", "한남동", "여의도동"]
all_news = []

for dong in target_dongs:
    print(f">> {dong} 상권 뉴스 수집 중...")
    news_data = fetch_dong_news(dong, keyword="외식 상권", display=50)
    all_news.extend(news_data)

# 6. DataFrame 변환 및 3년 이내 날짜 필터링
df_news = pd.DataFrame(all_news)

if not df_news.empty:
    # (1) pubDate 문자열을 datetime 객체로 변환 (RFC 2822 포맷 자동 변환)
    df_news['pubDate_dt'] = pd.to_datetime(df_news['pubDate'], errors='coerce')
    
    # (2) 현재 기준 3년 전 기준 시점 계산 (시간대 UTC 통일)
    cutoff_date = pd.Timestamp.now(tz=timezone.utc) - pd.DateOffset(years=3)
    
    # (3) 3년 이내 기사만 추출
    df_news = df_news[df_news['pubDate_dt'] >= cutoff_date].copy()
    
    # 정렬 및 불필요한 임시 날짜 컬럼 정리 (선택)
    df_news = df_news.sort_values(by='pubDate_dt', ascending=False).reset_index(drop=True)
    df_news = df_news.drop(columns=['pubDate_dt'])

# 7. 파일 저장
os.makedirs("data", exist_ok=True)
df_news.to_csv("data/seoul_dong_news.csv", index=False, encoding="utf-8-sig")
df_news.to_json("data/seoul_dong_news.json", orient="records", force_ascii=False, indent=4)

print(f"\n최근 3년 이내 수집 완료! 총 {len(df_news)}건 저장됨.")
print(df_news.groupby('dong').head(1)[['dong', 'title', 'pubDate']])

>> 망원동 상권 뉴스 수집 중...
>> 성수동 상권 뉴스 수집 중...


C:\Users\user\AppData\Local\Temp\ipykernel_12168\2306948467.py:17: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  clean_text = BeautifulSoup(clean_text, "html.parser").text


>> 연남동 상권 뉴스 수집 중...
>> 한남동 상권 뉴스 수집 중...
>> 여의도동 상권 뉴스 수집 중...

최근 3년 이내 수집 완료! 총 168건 저장됨.
    dong                                          title  \
0    성수동  [유통레이더] 입추 지나자 맨투맨 매출 338%↑…CJ온스타일, 가을 패션 ...   
1    망원동                외국인 관광객, 침체된 소상공인 경기에 '구원투수' 될까   
4    연남동        골든블루, '쿼츠 인 더 시티' 진행…매장별 맞춤 레시피로 경쟁력 제고   
6    한남동  [유통가 NOW] 우아한형제들, 파스쿠찌, 쿠쿠, 현대홈쇼핑, 바디프랜드, ...   
56  여의도동                           [중기 뉴스픽] 중기부·한유원·소진공   

                            pubDate  
0   Wed, 19 Aug 2026 13:28:00 +0900  
1   Wed, 19 Aug 2026 08:30:00 +0900  
4   Tue, 11 Aug 2026 18:16:00 +0900  
6   Mon, 10 Aug 2026 16:24:00 +0900  
56  Wed, 01 Apr 2026 17:26:00 +0900  
